# Prompt Repetition for RTL Languages: experiment runner

Replicates arXiv:2512.14982 (*Prompt Repetition Improves Non-Reasoning LLMs*) on 6 RTL languages
(pa, ur, ps, ar, fa, sd) with **8 open models** and the repo's **4 tests**
(`baseline`, `repetition`, `cross_lingual_t1`, `cross_lingual_t2`).

**Progress is never lost:**
* Every chunk of generations is appended to `results/raw/<model>/<lang>.jsonl` and fsync'd.
* With `HF_RESULTS_REPO` set, those files are mirrored to a **private** Hugging Face dataset every few minutes.
* Re-running this notebook on any machine restores from the mirror and skips everything already done.
  A crash or disconnect costs at most one chunk (~512 prompts).
* Each model runs in its own subprocess, so a crash in one model doesn't stop the rest and GPU memory is freed between models.

**Before running:** add a secret named `HF_TOKEN` (an HF *write* token) with access to the gated
Llama 3.x and Gemma 2 repos (accept their licenses on huggingface.co first).
* Kaggle: Add-ons → Secrets. Also enable Internet and pick **GPU T4 x2**.
* Colab: 🔑 Secrets panel. Pick an L4 or A100 runtime if available.

**Unattended runs:** on Kaggle use **Save Version → Save & Run All (Commit)**. It keeps running after
you close the browser, up to 12 h. If it hits the limit, just commit again and it resumes.

## 1. Configuration

In [ ]:
# Ordered small → large so results start arriving quickly.
MODELS = [
    "llama3.2-1b", "qwen2.5-1.5b", "gemma2-2b", "qwen2.5-3b",
    "llama3.2-3b", "mistral-7b-v0.3", "qwen2.5-7b", "llama3.1-8b",
]
LANGUAGES = ["pa", "ur", "ps", "ar", "fa", "sd"]

# Tests 1–2 now. For tests 3–4 later, add "cross_lingual_t1", "cross_lingual_t2":
# completed baseline/repetition rows are reused, only the new methods run.
METHODS = ["baseline", "repetition"]

# Tasks to leave out, as TASK or LANG:TASK.
SKIP = []

# Private HF dataset that mirrors results (created automatically). Strongly recommended:
# it's what lets a new Kaggle/Colab session pick up where the last one stopped.
HF_RESULTS_REPO = ""          # empty = "<your-hf-username>/prompt-repetition-results" (private)

REPO_URL = "https://github.com/FizzahNasir/prompt_repitition_research.git"
BRANCH   = "kaggle-tests-1-2"

CHUNK = 512                   # prompts per checkpoint
# Stop cleanly this many hours after the notebook starts (Kaggle hard-stops at 12 h and
# can lose output when it does). None = no limit.
TIME_BUDGET_HOURS = 11.3
SMOKE_TEST = True             # run 20 prompts on the smallest model first

## 2. Platform, secrets and persistent storage

In [ ]:
import os, sys, subprocess, time
from pathlib import Path
NOTEBOOK_START = time.time()

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

def get_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception as e:
        print(f"secret {name} not available: {e}")
    return None

if ON_KAGGLE:
    WORK = Path("/tmp/work");       RESULTS_DIR = Path("/kaggle/working/results")
elif ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content");        RESULTS_DIR = Path("/content/drive/MyDrive/prompt_repetition_results")
else:
    WORK = Path.cwd().parent;       RESULTS_DIR = Path.cwd() / "results"
WORK.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = get_secret("HF_TOKEN")
if not HF_TOKEN:
    # Stop before spending GPU time: gated models and the results mirror both need it.
    raise SystemExit("HF_TOKEN secret missing — add it (Add-ons → Secrets), attach it, and run again.")
os.environ["HF_TOKEN"] = HF_TOKEN
if not HF_RESULTS_REPO:
    from huggingface_hub import whoami
    HF_RESULTS_REPO = f"{whoami(token=HF_TOKEN)['name']}/prompt-repetition-results"
os.environ["HF_RESULTS_REPO"] = HF_RESULTS_REPO
os.environ["PYTHONUNBUFFERED"] = "1"

platform = "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "local"
print(f"Platform: {platform}\nResults:  {RESULTS_DIR}\nHF mirror: {HF_RESULTS_REPO or '(disabled)'}")

## 3. Get the code and install dependencies

In [ ]:
REPO = WORK / "prompt_repitition_research"
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

# vLLM gives batched inference (~10–50x faster than one-at-a-time generate).
# If it can't install or load a model, pr_runner falls back to batched transformers.
# Default vLLM wheels need NVIDIA driver >= 580 (CUDA 13); older drivers get the CUDA 12.8 build.
VLLM_VERSION = "0.30.0"
driver = subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                        capture_output=True, text=True).stdout.strip().splitlines()[0]
print("NVIDIA driver:", driver)
extra = [] if int(driver.split(".")[0]) >= 580 else ["--extra-index-url", "https://download.pytorch.org/whl/cu128"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}", *extra,
                "openpyxl", "rapidfuzz", "scipy", "pandas", "huggingface_hub"], check=False)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 4. Plan: jobs per model (no GPU used)

In [ ]:
STOP_AT = NOTEBOOK_START + TIME_BUDGET_HOURS * 3600 if TIME_BUDGET_HOURS else None

def runner(model, *extra, out=RESULTS_DIR):
    return [sys.executable, "pr_runner.py", "--model", model, "--out", str(out),
            *(["--stop-at", str(STOP_AT)] if STOP_AT else []),
            "--languages", *LANGUAGES, "--methods", *METHODS, "--chunk", str(CHUNK),
            *(["--skip", *SKIP] if SKIP else []), *extra]

subprocess.run(runner(MODELS[0], "--dry-run"), cwd=REPO)

## 5. Smoke test (20 prompts, separate folder)

In [ ]:
import json
if SMOKE_TEST:
    smoke = WORK / "smoke"
    r = subprocess.run(runner(MODELS[0], "--limit", "20", "--languages", "ar", "--hf-repo", "",
                              out=smoke), cwd=REPO)
    for line in (smoke / "raw" / MODELS[0] / "ar.jsonl").read_text(encoding="utf-8").splitlines()[:6]:
        row = json.loads(line)
        print(row["task"], row["method"], "| gold:", row["correct_answer"], "| ok:", row["is_correct"],
              "|", (row["response"] or row.get("error") or "")[:80].replace("\n", " "))

## 6. Run all models (resume-safe — re-run this cell any time)

Small models (weights ≤ ~7 GB) run as **one process per GPU**, each on half the prompts (about 2x throughput
on Kaggle's T4 x2). Larger models, and Gemma-2 (which must run in fp32 on T4), run as one process split across
both GPUs. Each process writes its own file, and resume reads all of them.

In [ ]:
import threading, torch

N_GPUS = torch.cuda.device_count()
BF16 = N_GPUS > 0 and torch.cuda.get_device_capability(0)[0] >= 8
SIZE_B = {"qwen2.5-1.5b": 1.5, "qwen2.5-3b": 3.1, "qwen2.5-7b": 7.6, "mistral-7b-v0.3": 7.3,
          "gemma2-2b": 2.6, "llama3.2-1b": 1.2, "llama3.2-3b": 3.2, "llama3.1-8b": 8.0}

def weights_gb(model):
    fp32 = model.startswith("gemma2") and not BF16   # Gemma-2 overflows in fp16
    return SIZE_B[model] * (4 if fp32 else 2)

log_path = RESULTS_DIR / "run_log.txt"
log_lock = threading.Lock()

def stream(proc, tag):
    with open(log_path, "a", encoding="utf-8") as log:
        for line in proc.stdout:
            if "it/s]" in line or "Processed prompts" in line:
                continue  # drop progress-bar noise
            with log_lock:
                print(f"[{tag}] {line}", end="", flush=True)
                log.write(f"{tag}: {line}")

def run_model(model):
    shards = N_GPUS if (N_GPUS > 1 and weights_gb(model) <= 7) else 1
    procs = []
    for i in range(shards):
        env = dict(os.environ)
        cmd = runner(model)
        if shards > 1:
            env["CUDA_VISIBLE_DEVICES"] = str(i)
            cmd += ["--shard", f"{i}/{shards}"]
        p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        t = threading.Thread(target=stream, args=(p, f"{model}#{i}" if shards > 1 else model))
        t.start()
        procs.append((p, t))
    codes = []
    for p, t in procs:
        p.wait(); t.join()
        codes.append(p.returncode)
    return codes

status = {}
for model in MODELS:
    for attempt in (1, 2):
        print(f"\n{'=' * 70}\n{model}  (attempt {attempt})\n{'=' * 70}", flush=True)
        codes = run_model(model)
        status[model] = codes
        if all(c in (0, 3) for c in codes):   # 3 = time budget reached
            break
        time.sleep(30)                       # transient failure: retry once; resume skips done work
    if STOP_AT and time.time() > STOP_AT:
        print("Time budget reached — all progress is saved. Run the notebook again to continue.")
        break
print("\nExit codes (0 = done, 3 = stopped for time budget):", status)

## 7. Progress

In [ ]:
import pandas as pd
sys.path.insert(0, str(REPO))
from pr_runner import read_jsonl

rows = []
for model in MODELS:
    for lang in LANGUAGES:
        files = [f for f in (RESULTS_DIR / "raw" / model).glob(f"{lang}*.jsonl")
                 if f.name == f"{lang}.jsonl" or f.name.startswith(f"{lang}.shard")]
        merged = {r["key"]: r for f in files for r in read_jsonl(f) if r.get("method") in METHODS}
        data = list(merged.values())
        rows.append({"model": model, "lang": lang,
                     "done": sum(1 for r in data if not r.get("error")),
                     "errors": sum(1 for r in data if r.get("error")),
                     "too_long": sum(1 for r in data if r.get("skipped"))})
prog = pd.DataFrame(rows)
display(prog.pivot(index="model", columns="lang", values="done").loc[MODELS])
if prog["errors"].sum() or prog["too_long"].sum():
    display(prog[(prog.errors > 0) | (prog.too_long > 0)])

## 8. Score + statistics (McNemar, no correction, p < 0.1)

In [ ]:
subprocess.run([sys.executable, "score_results.py", "--results", str(RESULTS_DIR)], cwd=REPO)
acc = pd.read_csv(RESULTS_DIR / "analysis" / "accuracy_table.csv")
display(acc.pivot_table(index=["language", "task", "scenario"], columns=["model", "method"],
                        values="accuracy").round(3))

## 9. Download everything (optional)

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK / "prompt_repetition_results"), "zip", RESULTS_DIR)
print(archive)
if ON_COLAB:
    from google.colab import files
    files.download(archive)